# Query Rewriting for Better Retrieval

## 1. Project Overview

**Task:** Use an LLM to rewrite vague, ambiguous, or poorly-formed user questions into precise, retrieval-friendly queries — then compare retrieval quality before and after rewriting.

**Why this matters:** Users rarely phrase questions the way information is stored. They ask vague things like *"how does that thing with the containers work?"* when the document uses terms like *"Docker container orchestration."* Query rewriting bridges this vocabulary gap, and it is one of the highest-impact, lowest-effort improvements you can make to any RAG system.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — LLM for query rewriting and answer generation
- `HuggingFaceEmbeddings` (`all-MiniLM-L6-v2`) — embedding model
- `ChromaDB` — vector store for dense retrieval

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Understand **why** raw user queries fail at retrieval |
| 2 | Build an LLM-based **query rewriting** pipeline |
| 3 | Compare retrieval **Recall@k and MRR** with vs without rewriting |
| 4 | Implement **multi-query expansion** — generate multiple search queries from one question |
| 5 | Analyze **failure modes** — when rewriting hurts instead of helps |
| 6 | Learn practical **prompt engineering** for query transformation |

## 3. Why Do Vague Queries Fail?

Retrieval systems (both sparse and dense) match the **words or meaning of the query** against stored documents. Vague queries fail because:

| Problem | Example | What Goes Wrong |
|---------|---------|----------------|
| **Missing key terms** | "how does that memory thing work" | No term matches "RAM", "cache", or "virtual memory" |
| **Ambiguity** | "tell me about Python" | Could be the language, the snake, or Monty Python |
| **Conversational phrasing** | "so like, what's the deal with containers" | Filler words dilute the signal |
| **Pronouns / references** | "how does it scale" | "it" has no referent without prior context |
| **Too broad** | "explain networking" | Matches too many unrelated documents |
| **Too narrow jargon** | "K8s HPA cpu-based autoscaling" | Misses docs that use full words |

**Query rewriting** uses an LLM to transform the raw question into a precise, self-contained query that retrieves better passages.

## 4. Rewriting Strategies

```
USER QUESTION (vague)
       │
       ▼
  ┌─────────────────────────────────────────────┐
  │  STRATEGY 1: Single Rewrite                 │
  │  "how does that memory thing work"           │
  │   → "How does computer memory (RAM) work     │
  │      and what is the difference between RAM  │
  │      and cache memory?"                      │
  └─────────────────────────────────────────────┘
       │
       ▼
  ┌─────────────────────────────────────────────┐
  │  STRATEGY 2: Multi-Query Expansion           │
  │  "explain database scaling"                  │
  │   → Q1: "horizontal vs vertical database     │
  │          scaling strategies"                  │
  │   → Q2: "database sharding and partitioning" │
  │   → Q3: "read replicas and load balancing    │
  │          for databases"                      │
  └─────────────────────────────────────────────┘
       │
       ▼
  ┌─────────────────────────────────────────────┐
  │  STRATEGY 3: Step-Back Prompting             │
  │  "why does my Docker container keep crashing"│
  │   → "What are common causes of Docker        │
  │      container failures and how to debug     │
  │      container crashes?"                     │
  └─────────────────────────────────────────────┘
       │
       ▼
  RETRIEVER  →  better passages  →  better answers
```

## 5. Pipeline Architecture

```
  User Question
       │
  ┌────┴────────────────────────────────────────┐
  │  PATH A: No Rewriting (baseline)             │
  │  raw query → retriever → passages → LLM     │
  └──────────────────────────────────────────────┘
       │
  ┌────┴────────────────────────────────────────┐
  │  PATH B: Single Rewrite                      │
  │  raw query → LLM rewrite → retriever         │
  │           → passages → LLM answer            │
  └──────────────────────────────────────────────┘
       │
  ┌────┴────────────────────────────────────────┐
  │  PATH C: Multi-Query Expansion               │
  │  raw query → LLM generates 3 queries         │
  │           → retrieve for each → union/dedup  │
  │           → rerank → LLM answer              │
  └──────────────────────────────────────────────┘
       │
  ┌────┴────────────────────────────────────────┐
  │  EVALUATION                                  │
  │  Compare Recall@k, MRR, answer quality       │
  └──────────────────────────────────────────────┘
```

## 6. Environment Setup

In [ ]:
# Uncomment if any packages are missing
# !pip install -q langchain langchain-ollama langchain-core langchain-community \
#     langchain-huggingface chromadb sentence-transformers

print("Dependencies: langchain, chromadb, sentence-transformers")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 7. Imports

## 8. Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from collections import Counter, defaultdict

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

LLM_MODEL = "qwen3.5:9b"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K = 5
TEMP_REWRITE = 0.2
TEMP_ANSWER = 0.1
TEMP_JUDGE = 0.0

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"Embeddings: {EMBED_MODEL}")
print(f"Retrieval top-k: {TOP_K}")

## 9. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.1) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 96):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Knowledge Base & Retriever

## 10. Build the Document Corpus

A synthetic knowledge base of 25 technical passages. These are written in a **formal, specific style** — exactly the kind of text that vague user questions fail to match.

In [ ]:
PASSAGES = [
    {"id": "P01", "topic": "containers",
     "text": "Docker containers encapsulate applications and their dependencies into isolated execution environments. Unlike virtual machines, containers share the host operating system kernel, which makes them significantly lighter in terms of resource consumption. Docker images are constructed from Dockerfiles, where each instruction creates a new layer in a union file system. The docker build command processes the Dockerfile instructions sequentially, caching intermediate layers for faster rebuilds."},
    {"id": "P02", "topic": "containers",
     "text": "Kubernetes (K8s) is a container orchestration platform that automates deployment, scaling, and management of containerized applications across clusters of machines. A Pod is the smallest deployable unit, containing one or more containers that share storage and network. Deployments declare the desired state, and the control plane continuously reconciles the actual state to match. Horizontal Pod Autoscaler (HPA) adjusts the number of pod replicas based on observed CPU utilization or custom metrics."},
    {"id": "P03", "topic": "databases",
     "text": "Relational databases organize data into tables with predefined schemas. SQL (Structured Query Language) provides operations: SELECT for querying, INSERT for adding records, UPDATE for modification, and DELETE for removal. Indexes are B-tree or hash-based data structures that accelerate lookups by maintaining sorted references to table rows. Proper indexing strategy is critical for query performance, but excessive indexes slow down write operations."},
    {"id": "P04", "topic": "databases",
     "text": "Database sharding partitions data across multiple database instances to achieve horizontal scalability. Common sharding strategies include range-based (partition by value ranges), hash-based (partition by hash of shard key), and directory-based (lookup table maps keys to shards). Sharding introduces complexity: cross-shard queries require scatter-gather operations, and rebalancing data when adding shards demands careful migration planning. Read replicas handle read-heavy workloads by duplicating data to secondary nodes."},
    {"id": "P05", "topic": "networking",
     "text": "The TCP/IP protocol stack operates in four layers. The application layer (HTTP, DNS, SMTP) provides user-facing services. The transport layer uses TCP for reliable, ordered delivery with three-way handshakes and acknowledgments, or UDP for fast, connectionless communication. The internet layer (IP) routes packets using destination IP addresses across heterogeneous networks. The link layer handles physical frame transmission over Ethernet, Wi-Fi, or cellular media."},
    {"id": "P06", "topic": "networking",
     "text": "Load balancers distribute incoming network traffic across multiple backend servers to ensure no single server is overwhelmed. Layer 4 (transport) load balancers route based on IP and port information. Layer 7 (application) load balancers inspect HTTP headers, cookies, or URL paths for intelligent routing. Common algorithms include round-robin, least connections, and weighted distribution. Health checks monitor backend availability, and unhealthy servers are automatically removed from the pool."},
    {"id": "P07", "topic": "security",
     "text": "OAuth 2.0 is an authorization framework enabling third-party applications to access user resources without exposing credentials. The authorization code flow involves redirecting the user to an authorization server, obtaining a time-limited authorization code, and exchanging it for an access token. Refresh tokens allow obtaining new access tokens without re-authentication. JSON Web Tokens (JWT) are commonly used as self-contained access tokens carrying claims about the user and permissions."},
    {"id": "P08", "topic": "security",
     "text": "Cryptographic hash functions produce fixed-size digests from arbitrary input data. SHA-256 generates a 256-bit hash with properties: deterministic computation, preimage resistance (cannot reverse the hash), collision resistance (infeasible to find two inputs with the same hash), and the avalanche effect (small input changes produce completely different outputs). Applications include password verification (storing hashes instead of plaintext), digital signatures, blockchain proof-of-work, and file integrity checking."},
    {"id": "P09", "topic": "ml_ops",
     "text": "Machine learning model deployment typically involves serving predictions through REST APIs or gRPC endpoints. Production architectures include model servers (TensorFlow Serving, TorchServe, or custom Flask/FastAPI applications), feature stores for consistent feature computation, model registries (MLflow) for version tracking, and monitoring systems (Prometheus, Grafana) for detecting data drift and performance degradation. Canary deployments route a fraction of traffic to new model versions before full rollout."},
    {"id": "P10", "topic": "ml_ops",
     "text": "Data drift occurs when the statistical distribution of production data diverges from the training data distribution. Common drift types include covariate shift (input feature distributions change), concept drift (the relationship between inputs and outputs changes), and prior probability shift (label distribution changes). Detection methods include the Population Stability Index (PSI), Kolmogorov-Smirnov test, and Page-Hinkley test. Automated retraining pipelines with drift detection triggers maintain model accuracy over time."},
    {"id": "P11", "topic": "version_control",
     "text": "Git uses a directed acyclic graph (DAG) to represent the commit history. Each commit is a snapshot of the entire repository, identified by a SHA-1 hash. Branches are lightweight pointers to commits. The git merge command combines divergent branches; fast-forward merges simply move the pointer, while three-way merges create a new commit reconciling differences. Rebase replays commits onto a new base, producing a linear history but rewriting commit hashes."},
    {"id": "P12", "topic": "version_control",
     "text": "CI/CD (Continuous Integration / Continuous Delivery) automates the software release process. CI involves automatically building and testing code on every commit, catching integration errors early. CD extends this by automating deployment to staging and production environments. Pipeline stages typically include linting, unit tests, integration tests, security scanning, artifact building, and deployment. Tools include GitHub Actions, GitLab CI, Jenkins, and CircleCI."},
    {"id": "P13", "topic": "caching",
     "text": "Redis is an in-memory data structure store supporting strings, lists, sets, sorted sets, hashes, and streams. Its single-threaded event loop and entirely in-memory architecture achieve sub-millisecond read/write latency. Redis provides two persistence mechanisms: RDB (point-in-time snapshots at configurable intervals) and AOF (append-only file logging every write operation). Redis Sentinel enables high availability through automatic leader election and failover when the primary node becomes unreachable."},
    {"id": "P14", "topic": "caching",
     "text": "Cache invalidation strategies determine when cached data should be refreshed. Time-to-live (TTL) expires entries after a fixed duration. Write-through caches update both the cache and the backing store on every write. Write-behind (write-back) caches batch writes to the backing store asynchronously for better performance but risk data loss. Cache-aside (lazy loading) populates the cache only on cache misses. The cache stampede problem occurs when many concurrent requests simultaneously encounter a cache miss."},
    {"id": "P15", "topic": "api_design",
     "text": "REST API design follows resource-oriented principles. URIs identify resources (nouns, not verbs): /users, /users/123, /users/123/orders. HTTP methods map to CRUD operations: GET (read), POST (create), PUT (replace), PATCH (partial update), DELETE (remove). Responses use standard HTTP status codes: 200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 404 Not Found, 500 Internal Server Error. Pagination, filtering, and sorting use query parameters."},
    {"id": "P16", "topic": "api_design",
     "text": "GraphQL is a query language for APIs that lets clients request exactly the data they need. Unlike REST, which returns fixed data structures from multiple endpoints, GraphQL exposes a single endpoint where clients specify their data requirements through typed queries. Schemas define types, fields, and relationships. Resolvers are functions that fetch data for each field. Benefits include eliminating over-fetching and under-fetching. N+1 query problems can be mitigated using DataLoader batching."},
    {"id": "P17", "topic": "messaging",
     "text": "Apache Kafka is a distributed event streaming platform designed for high-throughput, fault-tolerant data pipelines. Producers publish messages to topics, which are partitioned across brokers for parallelism. Consumer groups enable parallel consumption — each partition is read by exactly one consumer in the group. Kafka guarantees ordering within a partition. Messages are retained for a configurable period regardless of consumption, enabling replay. Kafka Connect integrates with external systems; Kafka Streams provides stream processing."},
    {"id": "P18", "topic": "messaging",
     "text": "Message queue patterns enable asynchronous communication between services. Point-to-point (queue) delivers each message to exactly one consumer, suitable for task distribution. Publish-subscribe (topic) delivers messages to all subscribers, enabling event broadcasting. Dead letter queues capture messages that fail processing after a configured number of retries, preventing poison messages from blocking the pipeline. Backpressure mechanisms (rate limiting, buffering) prevent producers from overwhelming slow consumers."},
    {"id": "P19", "topic": "observability",
     "text": "The three pillars of observability are metrics, logs, and traces. Metrics are numerical measurements over time (counters, gauges, histograms) collected by systems like Prometheus. Logs are timestamped text records of events, aggregated by tools like the ELK stack (Elasticsearch, Logstash, Kibana). Distributed traces track requests across microservice boundaries using correlation IDs, visualized by tools like Jaeger or Zipkin. Together, they provide comprehensive visibility into system behavior and enable root cause analysis."},
    {"id": "P20", "topic": "observability",
     "text": "Service Level Indicators (SLIs) measure specific aspects of service quality — typically availability, latency, throughput, and error rate. Service Level Objectives (SLOs) set target values for SLIs (e.g., 99.9% availability, p99 latency under 200ms). Service Level Agreements (SLAs) are contracts with customers specifying consequences for violating SLOs. Error budgets represent the allowed unreliability (e.g., 0.1% downtime per month). Teams consume error budget for deployments and use budget exhaustion as a signal to shift focus to reliability."},
    {"id": "P21", "topic": "testing",
     "text": "The testing pyramid recommends many unit tests (fast, isolated, testing individual functions), fewer integration tests (testing interactions between components), and very few end-to-end tests (testing complete user workflows through the full stack). Unit tests use mocks and stubs to isolate the unit under test. Integration tests verify that database queries, API calls, and message passing work correctly between real components. End-to-end tests are slow and brittle but catch issues that lower-level tests miss."},
    {"id": "P22", "topic": "testing",
     "text": "Property-based testing generates random inputs to verify that code satisfies invariants rather than checking specific cases. Libraries like Hypothesis (Python) and QuickCheck (Haskell) automatically generate edge cases including empty inputs, boundary values, and adversarial strings. When a test fails, the framework shrinks the input to find the minimal failing case. This approach especially excels at finding bugs in parsing, serialization, and mathematical code where edge cases are hard to enumerate manually."},
    {"id": "P23", "topic": "architecture",
     "text": "Microservices architecture decomposes applications into small, independently deployable services that communicate over networks. Each service owns its data, has its own deployment pipeline, and can be written in different programming languages. Benefits include independent scaling, fault isolation, and team autonomy. Challenges include distributed transactions, service discovery, network latency, and operational complexity. The strangler fig pattern enables gradual migration from monoliths by routing requests to new services incrementally."},
    {"id": "P24", "topic": "architecture",
     "text": "Event-driven architecture uses events (immutable records of state changes) as the primary communication mechanism between components. Event sourcing stores the full history of state changes rather than just current state, enabling audit trails and temporal queries. CQRS (Command Query Responsibility Segregation) separates read and write models — commands mutate state and publish events, while queries read from projections optimized for specific access patterns. This enables independent scaling of reads and writes."},
    {"id": "P25", "topic": "architecture",
     "text": "The CAP theorem states that a distributed system can guarantee at most two of three properties: Consistency (all nodes see the same data simultaneously), Availability (every request receives a response), and Partition Tolerance (the system continues operating despite network failures). In practice, network partitions are unavoidable, so the real tradeoff is between consistency and availability during partitions. CP systems (e.g., HBase) reject requests during partitions; AP systems (e.g., Cassandra) accept writes that may conflict."},
]

print(f"Total passages: {len(PASSAGES)}")
print(f"Topics: {dict(Counter(p['topic'] for p in PASSAGES))}")
print(f"Avg length: {sum(len(p['text'].split()) for p in PASSAGES) // len(PASSAGES)} words")

## 11. Build the Dense Retriever

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

chroma_client = chromadb.Client()
try:
    chroma_client.delete_collection("knowledge_base")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},
)

passage_texts = [p["text"] for p in PASSAGES]
passage_ids = [p["id"] for p in PASSAGES]
passage_metas = [{"passage_id": p["id"], "topic": p["topic"]} for p in PASSAGES]

vectors = embeddings.embed_documents(passage_texts)
collection.add(
    documents=passage_texts,
    embeddings=vectors,
    metadatas=passage_metas,
    ids=passage_ids,
)

print(f"Indexed {collection.count()} passages in ChromaDB")
print(f"Embedding dim: {len(vectors[0])}")

In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> list[dict]:
    """Dense retrieval via ChromaDB."""
    qv = embeddings.embed_query(query)
    results = collection.query(query_embeddings=[qv], n_results=top_k)
    hits = []
    for i in range(len(results["documents"][0])):
        hits.append({
            "passage_id": results["metadatas"][0][i]["passage_id"],
            "topic": results["metadatas"][0][i]["topic"],
            "text": results["documents"][0][i],
            "score": 1 - results["distances"][0][i],
        })
    return hits


# Quick test
test_hits = retrieve("container orchestration", top_k=3)
for h in test_hits:
    print(f"  {h['passage_id']} [{h['topic']:15s}] sim={h['score']:.3f} | {h['text'][:50]}...")

---

# Part B — Query Rewriting

## 12. The Problem: Vague Queries Fail

Let's first see **how badly** vague queries perform, so we have motivation.

In [ ]:
vague_examples = [
    {"vague": "how does that thing with the containers work",
     "expected": "P01"},
    {"vague": "so like what's the deal with making sure your app doesn't go down",
     "expected": "P06"},
    {"vague": "tell me about the message thing",
     "expected": "P17"},
    {"vague": "how do you know if your model is going bad",
     "expected": "P10"},
    {"vague": "what's that security token thing for logging in",
     "expected": "P07"},
]

print("VAGUE QUERY RETRIEVAL (no rewriting)")
print("=" * 100)
for ex in vague_examples:
    hits = retrieve(ex["vague"], top_k=5)
    retrieved_ids = [h["passage_id"] for h in hits]
    found = ex["expected"] in retrieved_ids
    rank = retrieved_ids.index(ex["expected"]) + 1 if found else None
    status = f"HIT @{rank}" if found else "MISS"
    print(f"\n  Q: {ex['vague']}")
    print(f"  Expected: {ex['expected']} | Status: {status}")
    print(f"  Retrieved: {retrieved_ids}")

## 13. Strategy 1 — Single Query Rewrite

The simplest approach: ask the LLM to reformulate the vague question into a **specific, precise** query.

In [ ]:
REWRITE_SYSTEM = """You are a search query optimizer. Your job is to rewrite vague or conversational
user questions into precise, specific queries that will retrieve relevant technical documents.

Rules:
1. Replace vague terms with specific technical terms
2. Remove filler words and conversational padding
3. Expand abbreviations and acronyms
4. Make implicit context explicit
5. Keep the rewritten query concise (1-2 sentences)
6. Do NOT answer the question — only rewrite it"""

REWRITE_PROMPT = """Rewrite this vague question into a precise search query.

VAGUE QUESTION: {question}

Return ONLY JSON:
{{"rewritten_query": "the improved query", "reasoning": "why this rewrite is better"}}"""


def rewrite_query(question: str) -> dict:
    """Single query rewrite via LLM."""
    resp = ask(
        REWRITE_PROMPT.format(question=question),
        system=REWRITE_SYSTEM,
        temperature=TEMP_REWRITE,
    )
    result = parse_json(resp)
    if result and "rewritten_query" in result:
        return result
    # Fallback: use raw response as rewritten query
    return {"rewritten_query": resp[:200], "reasoning": "fallback: raw LLM output"}


# Demo
demo = rewrite_query("how does that thing with the containers work")
print(f"Original:  how does that thing with the containers work")
print(f"Rewritten: {demo['rewritten_query']}")
print(f"Reasoning: {demo['reasoning']}")

## 14. Compare: Vague vs Rewritten Retrieval

In [ ]:
print("SINGLE REWRITE: SIDE-BY-SIDE COMPARISON")
print("=" * 110)

for ex in vague_examples:
    # Path A: raw vague query
    raw_hits = retrieve(ex["vague"], top_k=5)
    raw_ids = [h["passage_id"] for h in raw_hits]
    raw_found = ex["expected"] in raw_ids
    raw_rank = raw_ids.index(ex["expected"]) + 1 if raw_found else None

    # Path B: rewritten query
    rewrite = rewrite_query(ex["vague"])
    rw_hits = retrieve(rewrite["rewritten_query"], top_k=5)
    rw_ids = [h["passage_id"] for h in rw_hits]
    rw_found = ex["expected"] in rw_ids
    rw_rank = rw_ids.index(ex["expected"]) + 1 if rw_found else None

    raw_s = f"@{raw_rank}" if raw_found else "MISS"
    rw_s = f"@{rw_rank}" if rw_found else "MISS"
    improved = "IMPROVED" if (rw_rank or 99) < (raw_rank or 99) else ("SAME" if raw_rank == rw_rank else "WORSE")

    print(f"\n  Vague:    {ex['vague']}")
    print(f"  Rewritten: {rewrite['rewritten_query']}")
    print(f"  Expected: {ex['expected']} | Raw: {raw_s:>5} | Rewritten: {rw_s:>5} | {improved}")

## 15. Strategy 2 — Multi-Query Expansion

Instead of one rewritten query, generate **multiple diverse queries** from the original question. Retrieve for each, then merge results. This captures different facets of the question.

In [ ]:
MULTI_QUERY_SYSTEM = """You generate multiple precise search queries from a single user question.
Each query should approach the topic from a DIFFERENT angle to maximize retrieval coverage."""

MULTI_QUERY_PROMPT = """Generate 3 diverse search queries for the following question.
Each query should use different keywords and approach the topic from a different angle.

QUESTION: {question}

Return ONLY JSON:
{{"queries": ["query 1", "query 2", "query 3"]}}"""


def expand_multi_query(question: str) -> list[str]:
    """Generate multiple search queries from one question."""
    resp = ask(
        MULTI_QUERY_PROMPT.format(question=question),
        system=MULTI_QUERY_SYSTEM,
        temperature=TEMP_REWRITE,
    )
    result = parse_json(resp)
    if result and "queries" in result:
        return result["queries"][:3]
    # Fallback
    return [question]


def retrieve_multi_query(question: str, top_k: int = TOP_K) -> list[dict]:
    """Retrieve using multiple expanded queries and merge via score."""
    queries = expand_multi_query(question)
    seen = {}
    for q in queries:
        hits = retrieve(q, top_k=top_k)
        for h in hits:
            pid = h["passage_id"]
            if pid not in seen or h["score"] > seen[pid]["score"]:
                seen[pid] = h
    # Sort by best score and return top_k
    merged = sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:top_k]
    return merged, queries


# Demo
demo_q = "tell me about the message thing"
demo_results, demo_queries = retrieve_multi_query(demo_q, top_k=3)
print(f"Original: {demo_q}")
print(f"Expanded queries:")
for i, q in enumerate(demo_queries, 1):
    print(f"  Q{i}: {q}")
print(f"Retrieved: {[r['passage_id'] for r in demo_results]}")

## 16. Strategy 3 — Step-Back Prompting

Instead of making the query more specific, **step back** to a broader, more fundamental question. This works well when the user asks something too narrow or too specific to match.

In [ ]:
STEP_BACK_SYSTEM = """You transform specific, narrow questions into broader, more fundamental questions
that would retrieve background knowledge needed to answer the original question."""

STEP_BACK_PROMPT = """What is the broader, more fundamental question behind this specific question?
The step-back question should retrieve relevant background knowledge.

SPECIFIC QUESTION: {question}

Return ONLY JSON:
{{"step_back_query": "the broader question", "reasoning": "why this broader question helps"}}"""


def step_back_rewrite(question: str) -> dict:
    """Generate a step-back question."""
    resp = ask(
        STEP_BACK_PROMPT.format(question=question),
        system=STEP_BACK_SYSTEM,
        temperature=TEMP_REWRITE,
    )
    result = parse_json(resp)
    if result and "step_back_query" in result:
        return result
    return {"step_back_query": resp[:200], "reasoning": "fallback"}


# Demo
narrow_q = "why does K8s HPA not scale when CPU is high"
step_back = step_back_rewrite(narrow_q)
print(f"Narrow:    {narrow_q}")
print(f"Step-back: {step_back['step_back_query']}")
print(f"Reasoning: {step_back['reasoning']}")

# Compare retrieval
narrow_hits = retrieve(narrow_q, top_k=3)
broad_hits = retrieve(step_back["step_back_query"], top_k=3)
print(f"\nNarrow retrieved:    {[h['passage_id'] for h in narrow_hits]}")
print(f"Step-back retrieved: {[h['passage_id'] for h in broad_hits]}")

---

# Part C — Full Evaluation

## 17. Evaluation Dataset

20 vague queries paired with ground-truth passage IDs. These represent realistic user questions that a RAG system would receive.

In [ ]:
EVAL_SET = [
    # Very vague / conversational
    {"query": "how does that thing with the containers work",
     "expected_id": "P01", "vagueness": "high"},
    {"query": "what's the deal with making sure your app doesn't go down",
     "expected_id": "P06", "vagueness": "high"},
    {"query": "tell me about the message thing",
     "expected_id": "P17", "vagueness": "high"},
    {"query": "how do you know if your model is going bad",
     "expected_id": "P10", "vagueness": "high"},
    {"query": "what's that security token thing for logging in",
     "expected_id": "P07", "vagueness": "high"},
    {"query": "how do you store stuff really fast",
     "expected_id": "P13", "vagueness": "high"},
    {"query": "what happens if the network breaks",
     "expected_id": "P25", "vagueness": "high"},

    # Moderately vague
    {"query": "how do you split a big database across machines",
     "expected_id": "P04", "vagueness": "medium"},
    {"query": "what's the best way to test software",
     "expected_id": "P21", "vagueness": "medium"},
    {"query": "how do different services talk to each other",
     "expected_id": "P23", "vagueness": "medium"},
    {"query": "how to check if your system is working ok",
     "expected_id": "P19", "vagueness": "medium"},
    {"query": "what's the difference between the two API styles",
     "expected_id": "P16", "vagueness": "medium"},
    {"query": "how do you handle old cached data",
     "expected_id": "P14", "vagueness": "medium"},
    {"query": "how to put a model into production",
     "expected_id": "P09", "vagueness": "medium"},

    # Slightly vague but still imprecise
    {"query": "that thing where you record every state change",
     "expected_id": "P24", "vagueness": "medium"},
    {"query": "automatic build and deploy pipeline",
     "expected_id": "P12", "vagueness": "low"},
    {"query": "how git tracks code changes",
     "expected_id": "P11", "vagueness": "low"},
    {"query": "generating random test inputs automatically",
     "expected_id": "P22", "vagueness": "low"},
    {"query": "how do you keep promises to customers about uptime",
     "expected_id": "P20", "vagueness": "low"},
    {"query": "scaling containers automatically with Kubernetes",
     "expected_id": "P02", "vagueness": "low"},
]

print(f"Evaluation set: {len(EVAL_SET)} queries")
print(f"Vagueness: {dict(Counter(e['vagueness'] for e in EVAL_SET))}")

## 18. Run All Strategies and Compute Metrics

In [ ]:
def evaluate_strategy(eval_set, retrieval_fn, top_k=5):
    """Evaluate a retrieval strategy: Recall@k, MRR, per-query details."""
    hits = 0
    recip_ranks = []
    details = []

    for item in eval_set:
        result = retrieval_fn(item["query"], top_k=top_k)
        # Handle multi-query which returns (results, queries) tuple
        extra = None
        if isinstance(result, tuple):
            result, extra = result

        retrieved_ids = [r["passage_id"] for r in result]
        found_rank = None
        for rank, pid in enumerate(retrieved_ids, 1):
            if pid == item["expected_id"]:
                found_rank = rank
                break

        if found_rank:
            hits += 1
            recip_ranks.append(1.0 / found_rank)
        else:
            recip_ranks.append(0.0)

        details.append({
            "query": item["query"],
            "expected": item["expected_id"],
            "vagueness": item["vagueness"],
            "retrieved": retrieved_ids[:3],
            "hit": found_rank is not None,
            "rank": found_rank,
        })

    n = len(eval_set)
    return {
        "recall_at_k": hits / n,
        "mrr": sum(recip_ranks) / n,
        "n": n,
        "details": details,
    }


# Strategy wrappers
def strategy_raw(query, top_k=5):
    return retrieve(query, top_k)


def strategy_single_rewrite(query, top_k=5):
    rw = rewrite_query(query)
    return retrieve(rw["rewritten_query"], top_k)


def strategy_multi_query(query, top_k=5):
    return retrieve_multi_query(query, top_k)


def strategy_step_back(query, top_k=5):
    sb = step_back_rewrite(query)
    return retrieve(sb["step_back_query"], top_k)


print("Running evaluation for all 4 strategies...")
print("(This calls the LLM for each rewrite — may take a minute)\n")

eval_raw = evaluate_strategy(EVAL_SET, strategy_raw)
print(f"  [1/4] Raw queries done:      Recall@5 = {eval_raw['recall_at_k']:.1%}")

eval_single = evaluate_strategy(EVAL_SET, strategy_single_rewrite)
print(f"  [2/4] Single rewrite done:   Recall@5 = {eval_single['recall_at_k']:.1%}")

eval_multi = evaluate_strategy(EVAL_SET, strategy_multi_query)
print(f"  [3/4] Multi-query done:      Recall@5 = {eval_multi['recall_at_k']:.1%}")

eval_stepback = evaluate_strategy(EVAL_SET, strategy_step_back)
print(f"  [4/4] Step-back done:        Recall@5 = {eval_stepback['recall_at_k']:.1%}")

## 19. Overall Metrics Comparison

In [ ]:
print("OVERALL RETRIEVAL METRICS")
print("=" * 72)
print(f"{'Strategy':<25} {'Recall@5':>12} {'MRR':>12} {'Δ Recall':>12}")
print("-" * 72)

baseline = eval_raw["recall_at_k"]
for label, ev in [
    ("Raw (no rewriting)", eval_raw),
    ("Single rewrite", eval_single),
    ("Multi-query (3 queries)", eval_multi),
    ("Step-back prompting", eval_stepback),
]:
    delta = ev["recall_at_k"] - baseline
    delta_s = f"+{delta:.1%}" if delta > 0 else f"{delta:.1%}" if delta < 0 else "baseline"
    print(f"  {label:<23} {ev['recall_at_k']:>11.1%} {ev['mrr']:>11.3f} {delta_s:>12}")

## 20. Breakdown by Vagueness Level

In [ ]:
def metrics_by_vagueness(details):
    by_v = defaultdict(lambda: {"hits": 0, "rrs": [], "n": 0})
    for d in details:
        v = d["vagueness"]
        by_v[v]["n"] += 1
        if d["hit"]:
            by_v[v]["hits"] += 1
            by_v[v]["rrs"].append(1.0 / d["rank"])
        else:
            by_v[v]["rrs"].append(0.0)
    return {v: {"recall": d["hits"]/d["n"], "mrr": sum(d["rrs"])/d["n"], "n": d["n"]}
            for v, d in by_v.items()}


print("RECALL@5 BY VAGUENESS LEVEL")
print("=" * 85)

for vagueness in ["high", "medium", "low"]:
    print(f"\n  {vagueness.upper()} vagueness")
    print(f"    {'Strategy':<25} {'Recall':>10} {'MRR':>10}")
    for label, ev in [
        ("Raw", eval_raw),
        ("Single rewrite", eval_single),
        ("Multi-query", eval_multi),
        ("Step-back", eval_stepback),
    ]:
        by_v = metrics_by_vagueness(ev["details"])
        m = by_v.get(vagueness, {"recall": 0, "mrr": 0})
        print(f"    {label:<25} {m['recall']:>9.1%} {m['mrr']:>9.3f}")

## 21. Per-Query Winner Analysis

In [ ]:
print("PER-QUERY COMPARISON")
print("=" * 120)
print(f"  {'Query':<48} {'Expect':>6} {'Raw':>5} {'RW':>5} {'MQ':>5} {'SB':>5} {'Best':>7}")
print("-" * 120)

strategy_wins = {"raw": 0, "single": 0, "multi": 0, "stepback": 0}

for rd, sd, md, sbd in zip(
    eval_raw["details"], eval_single["details"],
    eval_multi["details"], eval_stepback["details"],
):
    r_rank = rd["rank"] or 99
    s_rank = sd["rank"] or 99
    m_rank = md["rank"] or 99
    sb_rank = sbd["rank"] or 99

    best = min(r_rank, s_rank, m_rank, sb_rank)
    if best == 99:
        winner = "NONE"
    elif r_rank == best:
        winner = "Raw"
        strategy_wins["raw"] += 1
    elif s_rank == best:
        winner = "Rewrite"
        strategy_wins["single"] += 1
    elif m_rank == best:
        winner = "Multi"
        strategy_wins["multi"] += 1
    else:
        winner = "StepBk"
        strategy_wins["stepback"] += 1

    fmt = lambda x: f"@{x}" if x < 99 else "MISS"
    q_short = textwrap.shorten(rd["query"], width=46, placeholder="...")
    print(f"  {q_short:<48} {rd['expected']:>6} {fmt(r_rank):>5} {fmt(s_rank):>5} {fmt(m_rank):>5} {fmt(sb_rank):>5} {winner:>7}")

print(f"\nBest strategy counts: {strategy_wins}")

---

# Part D — RAG Answer Quality Comparison

## 22. Answer Generation: Raw vs Rewritten

In [ ]:
QA_SYSTEM = """You are a knowledgeable technical assistant. Answer the question using ONLY the
provided passages. Cite passages as [P01], [P02], etc. Be concise and accurate."""


def answer_with_retrieval(question: str, retriever_fn, method_name: str) -> dict:
    """Retrieve passages and generate an answer."""
    result = retriever_fn(question, top_k=3)
    if isinstance(result, tuple):
        result = result[0]

    context = "\n\n".join(f"[{h['passage_id']}] {h['text']}" for h in result)
    prompt = (
        f"QUESTION: {question}\n\n"
        f"PASSAGES:\n{context}\n\n"
        "Answer using the passages. Cite sources as [PXX]."
    )
    answer = ask(prompt, system=QA_SYSTEM, temperature=TEMP_ANSWER)
    return {
        "method": method_name,
        "question": question,
        "retrieved": [h["passage_id"] for h in result],
        "answer": answer,
    }


# Compare answers on 3 vague questions
test_questions = [
    "how does that thing with the containers work",
    "what's the deal with making sure your app doesn't go down",
    "how do you know if your model is going bad",
]

for q in test_questions:
    print(f"\n{'='*100}")
    print(f"Q: {q}")
    print(f"{'='*100}")

    for label, fn in [("Raw", strategy_raw), ("Single Rewrite", strategy_single_rewrite)]:
        result = answer_with_retrieval(q, fn, label)
        print(f"\n  [{label}]")
        print(f"  Retrieved: {result['retrieved']}")
        print(f"  Answer: {textwrap.shorten(result['answer'], width=130, placeholder='...')}")

## 23. LLM-as-Judge: Answer Quality Scoring

In [ ]:
JUDGE_SYSTEM = "You evaluate QA answer quality."

JUDGE_PROMPT = """Compare these two answers to the same question.

QUESTION: {question}

ANSWER A (no rewriting): {answer_a}

ANSWER B (with query rewriting): {answer_b}

Score each 1-5 on: relevance, accuracy, completeness.
Then pick a winner.

Return ONLY JSON:
{{"answer_a": {{"relevance": N, "accuracy": N, "completeness": N}},
  "answer_b": {{"relevance": N, "accuracy": N, "completeness": N}},
  "winner": "A or B or TIE",
  "explanation": "why"}}"""

print("LLM-AS-JUDGE COMPARISON")
print("=" * 90)

judge_questions = [
    "how does that thing with the containers work",
    "how do you know if your model is going bad",
    "what happens if the network breaks",
]

for q in judge_questions:
    raw_ans = answer_with_retrieval(q, strategy_raw, "raw")
    rw_ans = answer_with_retrieval(q, strategy_single_rewrite, "rewrite")

    resp = ask(
        JUDGE_PROMPT.format(
            question=q,
            answer_a=raw_ans["answer"][:350],
            answer_b=rw_ans["answer"][:350],
        ),
        system=JUDGE_SYSTEM,
        temperature=TEMP_JUDGE,
    )
    scores = parse_json(resp)
    if scores:
        print(f"\n  Q: {textwrap.shorten(q, width=60, placeholder='...')}")
        print(f"  Raw retrieved:  {raw_ans['retrieved'][:3]}")
        print(f"  RW retrieved:   {rw_ans['retrieved'][:3]}")
        a = scores.get("answer_a", {})
        b = scores.get("answer_b", {})
        print(f"  Raw scores:     rel={a.get('relevance','?')} acc={a.get('accuracy','?')} comp={a.get('completeness','?')}")
        print(f"  Rewrite scores: rel={b.get('relevance','?')} acc={b.get('accuracy','?')} comp={b.get('completeness','?')}")
        print(f"  Winner: {scores.get('winner', '?')} — {scores.get('explanation', '?')[:80]}")

---

# Part E — Failure Analysis & Edge Cases

## 24. When Rewriting Hurts

Rewriting is not always better. It can **introduce errors** when:
- The LLM misinterprets the intent and rewrites to a different topic
- The original query was already specific enough
- The LLM adds assumptions not present in the question

In [ ]:
edge_cases = [
    # Already specific — rewriting might distort
    {"query": "Redis RDB vs AOF persistence comparison",
     "expected": "P13", "risk": "over-rewrite"},
    # Ambiguous — LLM might guess wrong domain
    {"query": "how to handle errors",
     "expected": "P18", "risk": "wrong interpretation"},
    # Very short — insufficient signal for rewriting
    {"query": "scaling",
     "expected": "P04", "risk": "too vague to rewrite correctly"},
]

print("EDGE CASES: WHEN REWRITING CAN FAIL")
print("=" * 100)

for ex in edge_cases:
    rw = rewrite_query(ex["query"])

    raw_hits = retrieve(ex["query"], top_k=5)
    rw_hits = retrieve(rw["rewritten_query"], top_k=5)

    raw_ids = [h["passage_id"] for h in raw_hits]
    rw_ids = [h["passage_id"] for h in rw_hits]

    raw_found = ex["expected"] in raw_ids
    rw_found = ex["expected"] in rw_ids

    print(f"\n  Risk: {ex['risk']}")
    print(f"  Original:    {ex['query']}")
    print(f"  Rewritten:   {rw['rewritten_query']}")
    print(f"  Expected: {ex['expected']}")
    raw_s = "HIT" if raw_found else "MISS"
    rw_s = "HIT" if rw_found else "MISS"
    outcome = "HELPED" if rw_found and not raw_found else ("HURT" if raw_found and not rw_found else "SAME")
    print(f"  Raw: {raw_s} | Rewritten: {rw_s} | Outcome: {outcome}")

## 25. Safeguard: Hybrid Rewrite Strategy

Retrieve with **both** the original and rewritten query, then merge. This way, rewriting can only help, never hurt.

In [ ]:
def strategy_safe_rewrite(question: str, top_k: int = 5) -> list[dict]:
    """Hybrid: retrieve with both original AND rewritten query, merge results."""
    # Get results from original query
    raw_hits = retrieve(question, top_k=top_k)

    # Get results from rewritten query
    rw = rewrite_query(question)
    rw_hits = retrieve(rw["rewritten_query"], top_k=top_k)

    # Merge: keep best score for each passage
    seen = {}
    for h in raw_hits + rw_hits:
        pid = h["passage_id"]
        if pid not in seen or h["score"] > seen[pid]["score"]:
            seen[pid] = h

    return sorted(seen.values(), key=lambda x: x["score"], reverse=True)[:top_k]


# Evaluate safely
print("Running SAFE REWRITE evaluation (original + rewritten, merged)...")
eval_safe = evaluate_strategy(EVAL_SET, strategy_safe_rewrite)

print(f"\nSAFE REWRITE COMPARISON")
print("=" * 72)
print(f"{'Strategy':<25} {'Recall@5':>12} {'MRR':>12}")
print("-" * 72)
for label, ev in [
    ("Raw (baseline)", eval_raw),
    ("Single rewrite", eval_single),
    ("Multi-query", eval_multi),
    ("Step-back", eval_stepback),
    ("SAFE (raw + rewrite)", eval_safe),
]:
    print(f"  {label:<23} {ev['recall_at_k']:>11.1%} {ev['mrr']:>11.3f}")

## 26. Visualizing the Rewrite Effect

In [ ]:
print("REWRITE EFFECT: DETAILED BREAKDOWN")
print("=" * 90)

helped = 0
hurt = 0
no_change = 0

for rd, sd in zip(eval_raw["details"], eval_single["details"]):
    r_rank = rd["rank"] or 99
    s_rank = sd["rank"] or 99

    if s_rank < r_rank:
        effect = "HELPED"
        helped += 1
    elif s_rank > r_rank:
        effect = "HURT  "
        hurt += 1
    else:
        effect = "SAME  "
        no_change += 1

    fmt_r = f"@{r_rank}" if r_rank < 99 else "MISS"
    fmt_s = f"@{s_rank}" if s_rank < 99 else "MISS"
    q_short = textwrap.shorten(rd["query"], width=50, placeholder="...")
    print(f"  {effect} | Raw={fmt_r:>5} → RW={fmt_s:>5} | {q_short}")

print(f"\nSummary: Helped={helped} | Hurt={hurt} | Same={no_change}")
print(f"Net positive: {helped - hurt} queries improved")

---

# Part F — Why Query Rewriting Works

## 27. The Vocabulary Gap Problem

The core reason query rewriting helps is the **vocabulary gap** between how users ask questions and how documents store information:

```
USER says:     "that thing where you split your database"
DOCUMENT says:  "Database sharding partitions data across multiple instances"

Shared vocabulary:  "database" (1 word out of ~8)
After rewrite:      "How does database sharding and horizontal partitioning work?"
Shared vocabulary:  "database", "sharding", "partitioning" (3 key terms)
```

Even dense (embedding-based) retrieval suffers from this gap because:
- Embedding models are trained on well-formed text, not vague conversational fragments
- Short, vague queries produce embeddings that are far from any specific document
- The embedding space clusters around **concepts**, and vague queries land between clusters

### Why Each Strategy Helps Differently

| Strategy | What It Fixes | Best For |
|----------|--------------|----------|
| **Single Rewrite** | Vocabulary gap, conversational noise | Most queries — fast and effective |
| **Multi-Query** | Narrow focus, missing facets | Complex questions with multiple aspects |
| **Step-Back** | Over-specificity, missing context | Debugging questions, "why" questions |
| **Safe (Original + Rewrite)** | Nothing — it only adds, never removes | Production systems (risk-averse) |

### Cost-Benefit Analysis

| Factor | Impact |
|--------|--------|
| **LLM calls added** | 1 per query (single rewrite) to 1-3 per query (multi-query) |
| **Latency added** | 200-500ms for local LLM rewrite |
| **Recall improvement** | Typically +10-30% on vague queries |
| **Risk of harm** | Low with safe strategy (original + rewrite merged) |
| **Implementation effort** | ~20 lines of code for single rewrite |

In [ ]:
# Visual proof: cosine similarity of vague vs rewritten queries to target passage
print("EMBEDDING PROXIMITY: Vague vs Rewritten")
print("=" * 85)
print(f"  {'Query type':<12} {'Query':<48} {'Cosine sim':>10}")
print("-" * 85)

probe_cases = [
    ("how does that thing with the containers work", "P01"),
    ("tell me about the message thing", "P17"),
    ("how do you store stuff really fast", "P13"),
]

for vague_q, target_id in probe_cases:
    rw = rewrite_query(vague_q)
    target_text = next(p["text"] for p in PASSAGES if p["id"] == target_id)

    vague_vec = embeddings.embed_query(vague_q)
    rw_vec = embeddings.embed_query(rw["rewritten_query"])
    target_vec = embeddings.embed_query(target_text)

    # Cosine similarity
    import numpy as np
    def cosine_sim(a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    sim_vague = cosine_sim(vague_vec, target_vec)
    sim_rw = cosine_sim(rw_vec, target_vec)
    delta = sim_rw - sim_vague

    vague_short = textwrap.shorten(vague_q, 46, placeholder="...")
    rw_short = textwrap.shorten(rw["rewritten_query"], 46, placeholder="...")

    print(f"  {'Vague':<12} {vague_short:<48} {sim_vague:>10.4f}")
    print(f"  {'Rewritten':<12} {rw_short:<48} {sim_rw:>10.4f}  (Δ{delta:+.4f})")
    print(f"  Target: {target_id}")
    print()

## 28. Common Mistakes

| Mistake | Why It's Wrong | Better Approach |
|---------|---------------|----------------|
| Rewriting without fallback | LLM can misinterpret and send retrieval off-track | Use safe strategy: retrieve with BOTH original and rewritten |
| Over-rewriting specific queries | "Redis AOF" → "how does data persistence work" loses specificity | Classify query specificity first; skip rewrite if already specific |
| Not evaluating rewrite quality | You can't know if rewrites help without metrics | Always compare Recall@k with vs without rewriting |
| Using same temperature for rewrite and answer | Rewriting benefits from slight creativity; answers need precision | Rewrite temp 0.2-0.3, answer temp 0.0-0.1 |
| Ignoring rewrite latency | Each rewrite adds an LLM call | Cache rewrites for common queries; batch where possible |
| Multi-query without deduplication | Same passage retrieved 3x wastes context window | Deduplicate by passage ID, keep highest score |

## 29. Production Improvement Ideas

1. **Query classifier** — detect if the query is already specific (skip rewrite) or vague (apply rewrite)
2. **Rewrite cache** — cache LLM rewrites for common/repeated queries to avoid latency
3. **A/B testing** — serve some users with rewriting, some without, measure answer quality
4. **Rewrite + rerank** — rewrite the query, retrieve broadly, then rerank with a cross-encoder
5. **Conversation-aware rewriting** — include chat history so "it" and "that" get resolved
6. **Fine-tuned rewriter** — distill a small model specifically for query rewriting (faster than general LLM)
7. **User feedback loop** — if the user rephrases their question, use the rephrase to improve the rewriter

## 30. Exercises

### Exercise 1
Build a **query specificity classifier**: use the LLM to score each query 1-5 for specificity. Only apply rewriting when specificity < 3. Compare Recall@5 against always-rewrite and never-rewrite.

### Exercise 2
Implement **conversation-aware rewriting**: given a chat history (3 prior turns), rewrite the latest query resolving pronouns and references. Test with queries like "how does it handle failures" where "it" refers to Kafka from a previous turn.

### Exercise 3
Combine query rewriting with **BM25 + Dense hybrid retrieval**: rewrite the query, then retrieve using both BM25 and dense with RRF fusion. Measure if rewriting + hybrid beats either technique alone.

### Mini Challenge
Build a **rewrite quality evaluator**: for each rewritten query, measure (a) cosine similarity to the target passage embedding, (b) whether key technical terms from the passage appear in the rewrite. Correlate rewrite quality with retrieval success.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Knowledge base: {len(PASSAGES)} passages, {len(set(p['topic'] for p in PASSAGES))} topics")
print(f"Evaluation: {len(EVAL_SET)} vague queries with ground truth")
print(f"")
print(f"STRATEGY PERFORMANCE")
print(f"  {'Strategy':<25} {'Recall@5':>10} {'MRR':>10}")
print(f"  {'-'*45}")
for label, ev in [
    ("Raw (no rewriting)", eval_raw),
    ("Single rewrite", eval_single),
    ("Multi-query", eval_multi),
    ("Step-back", eval_stepback),
    ("Safe (raw + rewrite)", eval_safe),
]:
    print(f"  {label:<25} {ev['recall_at_k']:>9.1%} {ev['mrr']:>9.3f}")
print(f"")
print(f"Techniques implemented:")
print(f"  - Single query rewrite (LLM reformulation)")
print(f"  - Multi-query expansion (3 diverse queries per question)")
print(f"  - Step-back prompting (broader conceptual queries)")
print(f"  - Safe hybrid rewrite (original + rewritten, merged)")
print(f"  - Vocabulary gap analysis (cosine similarity comparison)")
print(f"  - LLM-as-judge answer quality scoring")

## 31. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Vague queries fail** because they share few words/concepts with the stored documents |
| 2 | **Query rewriting** bridges the vocabulary gap by adding specific technical terms |
| 3 | **Single rewrite** is the best effort-to-impact ratio — one LLM call, significant improvement |
| 4 | **Multi-query expansion** excels for complex questions with multiple facets |
| 5 | **Step-back prompting** helps when the user is too specific and needs broader context |
| 6 | **Safe rewriting** (original + rewritten, merged) eliminates the risk of rewriting making things worse |
| 7 | Rewriting helps **most for highly vague queries** and adds less value for already-specific ones |
| 8 | **Cosine similarity** between query and target embeddings visibly increases after rewriting |
| 9 | The **cost** is 1 extra LLM call per query — usually worth the recall improvement |
| 10 | In production, combine rewriting with **caching** and **specificity classification** to minimize overhead |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*